In [ ]:
# -*- coding: utf-8 -*-
"""
三方演化博弈完整实现（扩展版）
AI 企业 – 用户 – 政府监管

功能包括：
1. 符号形式的复制动态方程
2. 雅可比矩阵与局部稳定性分析
3. 多初始条件下的数值仿真
4. 三维相空间 (x,y,z) 演化轨迹可视化（400 dpi JPG）

"""

import numpy as np
import sympy as sp
import matplotlib.pyplot as plt
from scipy.integrate import odeint
from mpl_toolkits.mplot3d import Axes3D

# ================== 修复中文显示 ==================
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False


# ================== 符号定义 ==================
x, y, z = sp.symbols('x y z')
Bg, Dg, F, alpha, R1, R2, C1, C2, C3, V1, V2, L1, L2 = sp.symbols(
    'Bg Dg F alpha R1 R2 C1 C2 C3 V1 V2 L1 L2'
)

# ================== 三方复制动态方程 ==================
def replicator_dynamics():
    dx = x * (1 - x) * (y * (R1 + L1) + z * F * (1 + alpha) - C1)
    dy = y * (1 - y) * (x * (V1 + V2) - V2 - C2 + L2)
    dz = z * (1 - z) * (x * y * R2 + Bg + Dg - C3)
    return [dx, dy, dz]


# ================== 雅可比矩阵 ==================
def compute_jacobian():
    eqs = replicator_dynamics()
    J = sp.Matrix([
        [sp.simplify(sp.diff(eq, var)) for var in (x, y, z)]
        for eq in eqs
    ])
    return J


# ================== 稳定性分析 ==================
def analyze_stability(jacobian, params):
    equilibrium_points = [
        (0, 0, 0), (0, 0, 1), (0, 1, 0), (0, 1, 1),
        (1, 0, 0), (1, 0, 1), (1, 1, 0), (1, 1, 1)
    ]

    subs_dict = {
        Bg: params['Bg'], Dg: params['Dg'], F: params['F'],
        alpha: params['alpha'], R1: params['R1'], R2: params['R2'],
        C1: params['C1'], C2: params['C2'], C3: params['C3'],
        V1: params['V1'], V2: params['V2'], L1: params['L1'],
        L2: params['L2']
    }

    results = {}
    for point in equilibrium_points:
        Jp = jacobian.subs({x: point[0], y: point[1], z: point[2]}).subs(subs_dict)
        J_np = np.array(Jp.tolist(), dtype=float)
        eigvals = np.linalg.eigvals(J_np)
        results[point] = eigvals

    return results


# ================== 数值仿真 ==================
def dynamic_simulation(ic, t, params):
    subs_dict = {
        Bg: params['Bg'], Dg: params['Dg'], F: params['F'],
        alpha: params['alpha'], R1: params['R1'], R2: params['R2'],
        C1: params['C1'], C2: params['C2'], C3: params['C3'],
        V1: params['V1'], V2: params['V2'], L1: params['L1'],
        L2: params['L2']
    }

    dx, dy, dz = replicator_dynamics()

    dx_f = sp.lambdify((x, y, z), dx.subs(subs_dict), 'numpy')
    dy_f = sp.lambdify((x, y, z), dy.subs(subs_dict), 'numpy')
    dz_f = sp.lambdify((x, y, z), dz.subs(subs_dict), 'numpy')

    def system(state, t):
        return [dx_f(*state), dy_f(*state), dz_f(*state)]

    return odeint(system, ic, t)


# ================== 三维相空间轨迹绘制 ==================
def plot_3d_trajectories(initial_conditions, t, params,
                         filename="3D_trajectory.jpg"):
    fig = plt.figure(figsize=(12, 10))
    ax = fig.add_subplot(111, projection='3d')

    for ic in initial_conditions:
        traj = dynamic_simulation(ic, t, params)
        ax.plot(
            traj[:, 0],
            traj[:, 1],
            traj[:, 2],
            lw=2,
            label=f'IC={ic}'
        )

    ax.set_xlabel('AI Enterprise Strategy $x$', fontsize=18)
    ax.set_ylabel('User Strategy $y$', fontsize=18)
    ax.set_zlabel('Government Strategy $z$', fontsize=18)

    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_zlim(0, 1)

    ax.view_init(elev=25, azim=45)
    ax.legend(fontsize=12)
    ax.grid(True)

    plt.tight_layout()
    plt.savefig(filename, dpi=400, format='jpg', bbox_inches='tight')
    plt.close()


# ================== 主程序 ==================
if __name__ == "__main__":

    # -------- 参数设定 --------
    params = {
        'Bg': 6,
        'Dg': 5,
        'F': 2,
        'alpha': 0.5,
        'R1': 3,
        'R2': 1,
        'C1': 10,
        'C2': 5,
        'C3': 9,
        'V1': 2,
        'V2': 4,
        'L1': 3,
        'L2': 10
    }

    # -------- 时间尺度 --------
    t = np.linspace(0, 20, 300)

    # -------- 多初始策略配置 --------
    initial_conditions = [
        (0.1, 0.1, 0.1),
        (0.9, 0.1, 0.1),
        (0.1, 0.9, 0.1),
        (0.1, 0.1, 0.9),
        (0.5, 0.5, 0.5),
        (0.8, 0.6, 0.3),
        (0.3, 0.7, 0.8)
    ]

    # -------- 三维相空间轨迹 --------
    plot_3d_trajectories(
        initial_conditions,
        t,
        params,
        filename="three_party_3D_phase_trajectory.jpg"
    )
